# 11 — Answer scorer beyond school science

The answer scorer (`ml_pipeline/answer_scorer.py`) ranks a typed or candidate answer against the source text: it is the
cheap first judge in typed-answer grading, before any model request. It was trained on SciQ (school science), and its
word TF-IDF vocabulary was learnt from SciQ too. On university security, computing and business questions about two thirds
of the words fall outside that vocabulary (against one third on SciQ itself), so their similarity to the source comes
out as zero. Students bring exactly those subjects.

**Data.** MMLU (Hendrycks et al. 2021, *Measuring Massive Multitask Language Understanding*, MIT licence), test questions of
seven subjects: computer security, college computer science, machine learning, business ethics, management, marketing,
professional accounting. In the app the scorer always reads an answer against source text, so each question gets a
support passage: the lead of the Wikipedia article a search for the question and its answer finds (CC BY-SA; fetched once
by `ml_pipeline/mmlu_support.py`, cached in `datasets/raw/MMLU/`, not committed). Questions are split 50/50 by a hash of the
question text, so none is in both halves.

**Metric.** As in notebook 03: accuracy@1, the share of questions whose correct option the scorer ranks first among the
four (chance 0.25). SciQ must not get worse.

**Candidates.**
- `current` — the shipped model (word vocabulary, SciQ only).
- `word · SciQ` — the same recipe retrained, to separate recipe from luck.
- `word · SciQ+MMLU` — word vocabulary learnt from SciQ plus the MMLU train half.
- `char · SciQ` — hashed character 3–5-grams (no learnt vocabulary), SciQ only.
- `char · SciQ+MMLU` — character n-grams, trained on both.

All use the shipped classifier type (logistic regression).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from ml_pipeline.notebook_setup import bootstrap
bootstrap()
import pandas as pd
from sklearn.linear_model import LogisticRegression
from ml_pipeline import config, mmlu_support
from ml_pipeline.answer_scorer import AnswerScorer
from ml_pipeline.dataset_loader import load_scorer_training_data
from ml_pipeline.features.answer_features import expand_candidates
from ml_pipeline.train_models import accuracy_at_1

sciq = load_scorer_training_data()
mmlu = mmlu_support.with_support(mmlu_support.to_sciq_shape(mmlu_support.load()),
                                 fetch=lambda q: '', cache=mmlu_support.load_cache())
print(f"SciQ rows {len(sciq):,}; MMLU questions with a passage {len(mmlu)}")
print(mmlu.groupby(['subject', 'split']).size().unstack())

SciQ rows 12,252; MMLU questions with a passage 710
split                     test  train
subject                              
business_ethics             34     36
college_computer_science    43     28
computer_security           27     37
machine_learning            36     40
management                  36     55
marketing                   97    100
professional_accounting     60     81


Out-of-vocabulary share of the shipped model's word vocabulary, on questions plus answers, to confirm the gap on this
exact set.

In [2]:
current = AnswerScorer.load(config.MODELS_DIR / 'answer_scorer.joblib')
vec = current.feature_builder.vectorizer
an, vocab = vec.build_analyzer(), vec.vocabulary_
def oov(texts):
    toks = [t for x in texts for t in an(str(x))]
    return 1 - sum(t in vocab for t in toks) / max(len(toks), 1)
sci_test = sciq[sciq.split == 'test']
rows = {'SciQ test': oov(sci_test.question + ' ' + sci_test.correct_answer)}
for s, g in mmlu.groupby('subject'):
    rows[s] = oov(g.question + ' ' + g.correct_answer)
pd.Series(rows).round(3).to_frame('out of vocabulary')

,out of vocabulary
SciQ test,0.369
business_ethics,0.659
college_computer_science,0.627
computer_security,0.668
machine_learning,0.611
management,0.643
marketing,0.631
professional_accounting,0.664


## Train and compare

In [3]:
def long(df):
    return expand_candidates(df.assign(split=df['split']))

sci_long = long(sciq)
sci_train, sci_test_long = sci_long[sci_long.split == 'train'], sci_long[sci_long.split == 'test']
mm_long = long(mmlu)
mm_train, mm_test = mm_long[mm_long.split == 'train'], mm_long[mm_long.split == 'test']
# Question ids must not collide once the two sets are stacked.
mixed_train = pd.concat([sci_train, mm_train.assign(question_id=mm_train.question_id + 10**6)], ignore_index=True)

def logreg():
    return LogisticRegression(class_weight='balanced', max_iter=1000, random_state=config.RANDOM_SEED)

models = {'current': current,
          'word · SciQ': AnswerScorer(logreg(), 'word').fit(sci_train),
          'word · SciQ+MMLU': AnswerScorer(logreg(), 'word').fit(mixed_train),
          'char · SciQ': AnswerScorer(logreg(), 'char').fit(sci_train),
          'char · SciQ+MMLU': AnswerScorer(logreg(), 'char').fit(mixed_train)}

table = {}
for name, m in models.items():
    row = {'SciQ test': accuracy_at_1(m, sci_test_long), 'MMLU test': accuracy_at_1(m, mm_test)}
    for s, g in mm_test.groupby(mm_test.question.map(dict(zip(mmlu.question, mmlu.subject)))):
        row[s] = accuracy_at_1(m, g)
    table[name] = row
results = pd.DataFrame(table).T.round(3)
results

,SciQ test,MMLU test,business_ethics,college_computer_science,computer_security,machine_learning,management,marketing,professional_accounting
current,0.906,0.291,0.212,0.326,0.407,0.194,0.472,0.316,0.167
word · SciQ,0.906,0.291,0.212,0.326,0.407,0.194,0.472,0.316,0.167
word · SciQ+MMLU,0.911,0.330,0.303,0.349,0.296,0.194,0.528,0.378,0.233
char · SciQ,0.926,0.309,0.212,0.279,0.407,0.194,0.417,0.347,0.283
char · SciQ+MMLU,0.930,0.333,0.333,0.302,0.333,0.222,0.444,0.378,0.283


In [4]:
print('MMLU test questions:', mm_test.question_id.nunique())
best = results['MMLU test'].idxmax()
print('best on MMLU:', best, '| its SciQ change vs current:',
      round(results.loc[best, 'SciQ test'] - results.loc['current', 'SciQ test'], 3))

MMLU test questions: 333
best on MMLU: char · SciQ+MMLU | its SciQ change vs current: 0.024


## When may the scorer decide on its own?

Even the best candidate is near chance on university questions, so a better model is not enough: in typed-answer grading
the scorer must stop calling answers right or wrong when it is unsure. A first try gated on the share of words outside the
SciQ vocabulary; it caught almost nothing (single words of university questions are mostly ordinary English; the
two-thirds out-of-vocabulary figure above comes from word pairs) and the kept questions were no better ranked, so it was
dropped. The scorer's own confidence works: the gap between its top and second-ranked option. The table shows, per
threshold, how many questions it would still settle alone and how often it is right on those.

In [5]:
best_model = models['char · SciQ+MMLU']
def margins(long_df):
    t = long_df.assign(p=best_model.predict_proba(long_df))
    rows = []
    for _, g in t.groupby('question_id'):
        g = g.sort_values('p', ascending=False)
        rows.append((g.p.iloc[0] - g.p.iloc[1], int(g.is_correct.iloc[0] == 1)))
    return pd.DataFrame(rows, columns=['margin', 'hit'])
gaps = {'SciQ': margins(sci_test_long), 'MMLU': margins(mm_test)}
rows = []
for cut in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5):
    row = {'gap at least': cut}
    for name, d in gaps.items():
        kept = d[d.margin >= cut]
        row[f'{name} settled alone'] = round(len(kept) / len(d), 3)
        row[f'{name} right when settled'] = round(kept.hit.mean(), 3) if len(kept) else None
    rows.append(row)
pd.DataFrame(rows)

,gap at least,SciQ settled alone,SciQ right when settled,MMLU settled alone,MMLU right when settled
0,0.0,1.000,0.930,1.000,0.333
1,0.1,0.845,0.968,0.315,0.476
2,0.2,0.734,0.980,0.204,0.515
3,0.3,0.658,0.983,0.144,0.583
4,0.4,0.587,0.987,0.102,0.706
5,0.5,0.514,0.987,0.075,0.840


## Decision

The rule, fixed before the results: ship the candidate with the best MMLU test accuracy if it beats `current` there by at
least 0.03 and loses no more than 0.01 on SciQ test; otherwise keep `current`.

## Limits

- The support passage is a Wikipedia lead found by searching the question and its answer, not the student's own lecture
  notes; it contains the answer more often than a real source might, which helps every candidate equally.
- MMLU options are short and often numeric (accounting), unlike the app's generated questions.
- About 250 test questions across seven subjects: per-subject figures rest on 13–70 questions each.